##L02: Delta Lake fundamentals on a practice table

###Step 1. Create and change a Delta table
Use a SQL notebook (or %sql cells). Every command creates a new table version.

In [0]:
CREATE OR REPLACE TABLE da.default.delta_practice (id INT, city STRING, fare DOUBLE);
INSERT INTO da.default.delta_practice VALUES (1,'Brisbane',12.5),(2,'Sydney',20.0),(3,'Perth',15.0);
UPDATE da.default.delta_practice SET fare = fare * 1.1 WHERE city = 'Perth';
DELETE FROM da.default.delta_practice WHERE id = 2;
SELECT * FROM da.default.delta_practice;

###Step 2. Read the history
Each row is one commit. Note the operation, version and operationMetrics (rows added, files added and removed).

In [0]:
DESCRIBE HISTORY da.default.delta_practice;
DESCRIBE DETAIL da.default.delta_practice;

###Step 3. Time travel
Version 0 is the create, version 1 the insert. Query the table as it was after the insert. Then copy a timestamp from the history output and query by time.

In [0]:
SELECT * FROM da.default.delta_practice VERSION AS OF 1;
-- paste a timestamp from DESCRIBE HISTORY that is after the insert and before the update
SELECT * FROM da.default.delta_practice TIMESTAMP AS OF '2026-10-19T12:26:33';

###Step 4. Schema enforcement
Try to insert a row with the wrong shape. Delta must reject it. Read the error message and note the error class.

In [0]:
INSERT INTO da.default.delta_practice VALUES (4,'Cairns',9.0,'extra column');

###Step 5. Evolve the schema on purpose
Add a column with ALTER TABLE, then insert a row using it.

In [0]:
ALTER TABLE da.default.delta_practice ADD COLUMNS (country STRING);
INSERT INTO da.default.delta_practice VALUES (4,'Cairns',9.0,'Australia');
SELECT * FROM da.default.delta_practice;

###Step 6. Restore an earlier version
Roll back to version 1, then look at the history again. Restore is itself a new commit; history is never deleted.

In [0]:
RESTORE TABLE da.default.delta_practice TO VERSION AS OF 1;
SELECT * FROM da.default.delta_practice;
DESCRIBE HISTORY da.default.delta_practice;